# Feature importance example

In [6]:
import pprint

import numpy as np
import pandas as pd

from skrough.dataprep import prepare_factorized_data
from skrough.disorder_measures import conflicts_count, entropy, gini_impurity
from skrough.disorder_score import get_disorder_score_for_data
from skrough.feature_importance import get_feature_importance
from skrough.algorithms.reducts import get_approx_reduct_greedy_heuristic

## Dataset

Let's prepare a sample data set - "Play Golf Dataset".

In [2]:
df = pd.DataFrame(
    np.array(
        [
            ["sunny", "hot", "high", "weak", "no"],
            ["sunny", "hot", "high", "strong", "no"],
            ["overcast", "hot", "high", "weak", "yes"],
            ["rain", "mild", "high", "weak", "yes"],
            ["rain", "cool", "normal", "weak", "yes"],
            ["rain", "cool", "normal", "strong", "no"],
            ["overcast", "cool", "normal", "strong", "yes"],
            ["sunny", "mild", "high", "weak", "no"],
            ["sunny", "cool", "normal", "weak", "yes"],
            ["rain", "mild", "normal", "weak", "yes"],
            ["sunny", "mild", "normal", "strong", "yes"],
            ["overcast", "mild", "high", "strong", "yes"],
            ["overcast", "hot", "normal", "weak", "yes"],
            ["rain", "mild", "high", "strong", "no"],
        ],
        dtype=object,
    ),
    columns=["Outlook", "Temperature", "Humidity", "Wind", "Play"],
)
TARGET_COLUMN = "Play"
df

,Outlook,Temperature,Humidity,Wind,Play
0,sunny,hot,high,weak,no
1,sunny,hot,high,strong,no
2,overcast,hot,high,weak,yes
3,rain,mild,high,weak,yes
4,rain,cool,normal,weak,yes
5,rain,cool,normal,strong,no
6,overcast,cool,normal,strong,yes
7,sunny,mild,high,weak,no
8,sunny,cool,normal,weak,yes
9,rain,mild,normal,weak,yes


## Prepare data

Factorize dataset and obtain the sizes of feature domains.

In [3]:
x, x_counts, y, y_count = prepare_factorized_data(df, TARGET_COLUMN)
column_names = np.array([col for col in df.columns if col != TARGET_COLUMN])

print("Conditional data:")
print(x)
print()
print("Conditional data feature domain sizes:")
print(x_counts)
print()
print("Target data:")
print(y)
print()
print("Target data feature domain size:")
print(y_count)

Conditional data:
[[0 0 0 0]
 [0 0 0 1]
 [1 0 0 0]
 [2 1 0 0]
 [2 2 1 0]
 [2 2 1 1]
 [1 2 1 1]
 [0 1 0 0]
 [0 2 1 0]
 [2 1 1 0]
 [0 1 1 1]
 [1 1 0 1]
 [1 0 1 0]
 [2 1 0 1]]

Conditional data feature domain sizes:
[3 3 2 2]

Target data:
[0 0 1 1 1 0 1 0 1 1 1 1 1 0]

Target data feature domain size:
2


## Measure of disorder in the dataset - disorder score

In the context of the given dataset, a disorder score values is quantity that
characterizes a subset of features and, more or less, presents the disorder of
decisions in the equivalence classes induced by the subsets of features.

In most cases it is reasonable to assume that the disorder score function is monotonic
with respect to subset relation, i.e., for subsets of features $A \subseteq B$,
the disorder score for $A$ should be less or equal to that for $B$.

Attributes are given by their ordinal numbers.

Let's try three standard approaches, i.e., `conflicts_count`, `gini_impurity` and
`entropy`.

In [4]:
for disorder_function in [conflicts_count, entropy, gini_impurity]:
    print(disorder_function.__name__)
    for attrs in [[0], [0, 1], [0, 1, 3], [0, 1, 2, 3]]:
        print(
            f"disorder score for attrs {attrs}({column_names[attrs]}) = ",
            get_disorder_score_for_data(
                x=x,
                x_counts=x_counts,
                y=y,
                y_count=y_count,
                disorder_fun=disorder_function,
                attrs=attrs,
            ),
        )
    print()
    print()

conflicts_count
disorder score for attrs [0](['Outlook']) =  12.0
disorder score for attrs [0, 1](['Outlook' 'Temperature']) =  4.0
disorder score for attrs [0, 1, 3](['Outlook' 'Temperature' 'Wind']) =  0.0
disorder score for attrs [0, 1, 2, 3](['Outlook' 'Temperature' 'Humidity' 'Wind']) =  0.0


entropy
disorder score for attrs [0](['Outlook']) =  0.6935361388961918
disorder score for attrs [0, 1](['Outlook' 'Temperature']) =  0.4824919644402477
disorder score for attrs [0, 1, 3](['Outlook' 'Temperature' 'Wind']) =  0.0
disorder score for attrs [0, 1, 2, 3](['Outlook' 'Temperature' 'Humidity' 'Wind']) =  0.0


gini_impurity
disorder score for attrs [0](['Outlook']) =  0.34285714285714286
disorder score for attrs [0, 1](['Outlook' 'Temperature']) =  0.23809523809523808
disorder score for attrs [0, 1, 3](['Outlook' 'Temperature' 'Wind']) =  0.0
disorder score for attrs [0, 1, 2, 3](['Outlook' 'Temperature' 'Humidity' 'Wind']) =  0.0




## Computing reducts

In [15]:
result = get_approx_reduct_greedy_heuristic(
    x=x,
    y=y,
    disorder_fun=entropy,
    epsilon=0.2,
    n_reducts=10,
)
for approx_reduct in result:
    print([df.columns[i] for i in approx_reduct.attrs])

['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Wind', 'Temperature']
['Outlook', 'Wind', 'Humidity']
['Outlook', 'Wind', 'Humidity']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']
['Outlook', 'Humidity', 'Wind']


## Assessing feature importance

We can use the above disorder score functions for assessing the features, i.e.,
we can observe the disorder score change if a given feature is removed.

To follow a more realistic example, we can use an enseble of feature subsets, i.e.,
a family of subsets of all atributes, and not just a single subset of features,
computing the total or average disorder score change over several possible appearances
of the attribute in the ensemble elements.

In [5]:
attr_subset_ensemble = [
    [[0, 2], [0, 3], [0], [2, 3], [1, 2, 3]],
    [[0], [0, 1], [1, 2]],
    [[0, 1], [0, 3], [1, 2], [2, 3], [0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3]],
]
for disorder_function in [conflicts_count, entropy, gini_impurity]:
    print(disorder_function.__name__)
    for attr_subset in attr_subset_ensemble:
        print("feature importance for attribute subset ensemble: ")
        pprint.pprint(attr_subset, compact=True)
        print(
            get_feature_importance(
                x,
                x_counts,
                y,
                y_count,
                column_names,
                attr_subset,
                disorder_fun=disorder_function,
            )
        )
        print()
    print()
    print()

conflicts_count
feature importance for attribute subset ensemble: 
[[0, 2], [0, 3], [0], [2, 3], [1, 2, 3]]
        column  count  global_gain  avg_global_gain
0      Outlook    3.0         66.0        22.000000
1  Temperature    1.0          4.0         4.000000
2     Humidity    3.0         25.0         8.333333
3         Wind    3.0         24.0         8.000000

feature importance for attribute subset ensemble: 
[[0], [0, 1], [1, 2]]
        column  count  global_gain  avg_global_gain
0      Outlook    2.0         44.0             22.0
1  Temperature    2.0         17.0              8.5
2     Humidity    1.0          6.0              6.0
3         Wind    0.0          0.0              0.0

feature importance for attribute subset ensemble: 
[[0, 1], [0, 3], [1, 2], [2, 3], [0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3]]
        column  count  global_gain  avg_global_gain
0      Outlook    5.0         51.0             10.2
1  Temperature    5.0         25.0              5.0
2     Humidit